# Feature Engineering

> **DAY 2 — 모델 개발 및 평가**

DAY 1 EDA 결과를 기반으로 배터리 수명 예측 모델에 사용할 Feature를 생성한다.

### 목표
- 초기 Cycle 데이터만을 활용하여 Feature 생성
- EDA에서 확인한 주요 변수의 특성을 모델 입력값으로 변환
- Cell 단위 Feature Dataset 구축
- 이후 모델링 단계에서 사용할 데이터셋 준비

### 주요 Feature
- **Discharge Capacity (QD)**: 초기 용량 및 용량 감소 특성
- **ΔQ(V)**: 초기 Cycle 간 방전 곡선 변화
- **Internal Resistance (IR)**: 내부 저항 특성
- **Temperature**: 초기 Cycle의 온도 특성
- **Charging Condition**: 충전 프로토콜 및 C-rate 관련 특성

### Output
- Cell별 Feature Dataset
- Target: `cycle_life`

In [5]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

print("Feature Engineering 환경 설정 완료")

Feature Engineering 환경 설정 완료


## 1. Data Loading

원본 배터리 데이터를 불러와 Cycle 단위 DataFrame으로 변환한다.

In [6]:
from pathlib import Path

# 현재 노트북 위치 기준으로 프로젝트 루트 설정
PROJECT_ROOT = Path.cwd().parent

# 프로젝트 루트를 Python 경로에 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocess import load_battery_data, build_cycle_dataframe

print("preprocess 모듈 불러오기 완료")

preprocess 모듈 불러오기 완료


In [7]:
# 원본 배터리 데이터 경로
DATA_DIR = "/Users/seunghyunsuh/Downloads/archive"

# .mat 파일 로딩
mats = load_battery_data(DATA_DIR)

# Cycle 단위 DataFrame 생성
df = build_cycle_dataframe(mats)

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (2017-05-12_batchdata_updated_struct_errorcorrect.mat)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (2018-02-20_batchdata_updated_struct_errorcorrect.mat)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)


로딩 중... (2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (2018-04-12_batchdata_updated_struct_errorcorrect.mat)

총 4개 .mat 파일 로딩 완료

[통합 완료] 총 데이터 행 수: 117,775, 총 셀 개수: 141
포함된 배치 목록: <StringArray>
['2017-05-12', '2018-02-20', '2018-04-03', '2018-04-12']
Length: 4, dtype: str


In [8]:
print("DataFrame shape:", df.shape)
print("전체 Cell 수:", df["cell_id"].nunique())
print("Batch:", df["batch"].unique())

display(df.head())

DataFrame shape: (117775, 11)
전체 Cell 수: 141
Batch: <StringArray>
['2017-05-12', '2018-02-20', '2018-04-03', '2018-04-12']
Length: 4, dtype: str


,cell_id,batch,cycle,cycle_life,charging_policy,QD,QC,IR,Tavg,Tmax,chargetime
0,2017-05-12_cell_0,2017-05-12,1,1190.0,3_6C-80PER_3_6C,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2017-05-12_cell_0,2017-05-12,2,1190.0,3_6C-80PER_3_6C,0.272064,0.830364,0.016742,31.875011,35.652016,13.341250
2,2017-05-12_cell_0,2017-05-12,3,1190.0,3_6C-80PER_3_6C,0.281724,0.835266,0.016724,31.931490,35.692978,13.425777
3,2017-05-12_cell_0,2017-05-12,4,1190.0,3_6C-80PER_3_6C,0.283037,0.838370,0.016681,31.932603,35.680588,13.425167
4,2017-05-12_cell_0,2017-05-12,5,1190.0,3_6C-80PER_3_6C,0.279682,0.842304,0.016662,31.959322,35.728691,13.341442


## 2. Feature Generation

DAY 1 EDA 결과를 기반으로 초기 100 Cycle의 데이터를 활용하여 Cell 단위 Feature를 생성한다.

각 Cell의 시계열 데이터를 하나의 행으로 요약하여 모델 학습에 사용할 Feature Dataset을 구축한다.

In [9]:
# 초기 100 Cycle만 사용
early_df = df[
    (df["cycle"] >= 1) &
    (df["cycle"] <= 100)
].copy()

print("전체 행 수:", len(early_df))
print("전체 Cell 수:", early_df["cell_id"].nunique())

# Cell별 확보된 초기 Cycle 수 확인
cycle_counts = early_df.groupby("cell_id")["cycle"].nunique()

print("\nCell별 Cycle 수:")
print(cycle_counts.describe())

print("\n100 Cycle 미만인 Cell 수:", (cycle_counts < 100).sum())

전체 행 수: 14100
전체 Cell 수: 141

Cell별 Cycle 수:
count    141.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: cycle, dtype: float64

100 Cycle 미만인 Cell 수: 0


### 2.1 Discharge Capacity (QD) Features

초기 100 Cycle의 방전 용량(QD) 데이터를 이용하여 Cell별 용량 수준과 열화 경향을 나타내는 Feature를 생성한다.

- `mean_QD`: 초기 100 Cycle의 평균 방전 용량
- `std_QD`: 초기 100 Cycle의 방전 용량 변동성
- `QD_slope`: Cycle 증가에 따른 방전 용량 변화율

In [10]:
# QD Feature 생성 함수
def calculate_qd_features(group):
    # 0은 결측/미측정 값으로 처리
    valid = group[group["QD"] > 0].copy()

    if len(valid) < 2:
        return pd.Series({
            "mean_QD": np.nan,
            "std_QD": np.nan,
            "QD_slope": np.nan
        })

    # 평균 및 표준편차
    mean_qd = valid["QD"].mean()
    std_qd = valid["QD"].std()

    # Cycle에 따른 QD 변화율
    qd_slope = np.polyfit(
        valid["cycle"],
        valid["QD"],
        1
    )[0]

    return pd.Series({
        "mean_QD": mean_qd,
        "std_QD": std_qd,
        "QD_slope": qd_slope
    })


qd_features = (
    early_df
    .groupby("cell_id")
    .apply(calculate_qd_features)
    .reset_index()
)

print("QD Feature shape:", qd_features.shape)
display(qd_features.head())

QD Feature shape: (141, 4)


,cell_id,mean_QD,std_QD,QD_slope
0,2017-05-12_cell_0,0.280303,0.017804,-0.000074
1,2017-05-12_cell_1,0.280466,0.004920,0.000007
2,2017-05-12_cell_10,0.285288,0.005066,-0.000036
3,2017-05-12_cell_11,0.281397,0.005715,-0.000038
4,2017-05-12_cell_12,0.288248,0.004846,-0.000036


### 2.2 ΔQ(V) Features

DAY 1 EDA에서 Cycle 10과 Cycle 100의 `Qdlin` 차이가 Cycle Life와 높은 상관관계를 보였으므로, 동일한 방식으로 ΔQ(V) Feature를 생성한다.

- `deltaQ_mean`: ΔQ(V)의 평균
- `deltaQ_std`: ΔQ(V)의 표준편차
- `deltaQ_var`: ΔQ(V)의 분산
- `deltaQ_min`: ΔQ(V)의 최솟값
- `deltaQ_max`: ΔQ(V)의 최댓값
- `deltaQ_range`: ΔQ(V)의 범위

In [11]:
delta_qv_results = []

for filename, mat_data in mats.items():

    batch_name = filename.split("_")[0]

    main_key = (
        "batch"
        if "batch" in mat_data
        else [
            k for k in mat_data.keys()
            if not k.startswith("__")
        ][0]
    )

    cells = mat_data[main_key]["cycles"]

    for idx, cell in enumerate(cells):

        Qdlin = cell.get("Qdlin")

        # Cycle 100까지 존재하는 Cell만 사용
        if Qdlin is None or len(Qdlin) < 100:
            continue

        # Cycle 10 / Cycle 100
        Q10 = np.asarray(Qdlin[9], dtype=float).flatten()
        Q100 = np.asarray(Qdlin[99], dtype=float).flatten()

        # 두 curve의 길이 통일
        n = min(len(Q10), len(Q100))

        Q10 = Q10[:n]
        Q100 = Q100[:n]

        # NaN / inf 제거
        valid = np.isfinite(Q10) & np.isfinite(Q100)

        Q10 = Q10[valid]
        Q100 = Q100[valid]

        if len(Q10) < 10:
            continue

        # ΔQ(V) = Cycle 100 - Cycle 10
        delta_Q = Q100 - Q10

        cell_id = f"{batch_name}_cell_{idx}"

        delta_qv_results.append({
            "cell_id": cell_id,
            "deltaQ_mean": np.mean(delta_Q),
            "deltaQ_std": np.std(delta_Q),
            "deltaQ_var": np.var(delta_Q),
            "deltaQ_min": np.min(delta_Q),
            "deltaQ_max": np.max(delta_Q),
            "deltaQ_range": np.max(delta_Q) - np.min(delta_Q)
        })


delta_qv_features = pd.DataFrame(delta_qv_results)

print("ΔQ Feature shape:", delta_qv_features.shape)
display(delta_qv_features.head())

ΔQ Feature shape: (141, 7)


,cell_id,deltaQ_mean,deltaQ_std,deltaQ_var,deltaQ_min,deltaQ_max,deltaQ_range
0,2017-05-12_cell_0,-0.002873,0.003109,0.000010,-0.008460,0.000768,0.009228
1,2017-05-12_cell_1,-0.004100,0.003112,0.000010,-0.011004,-0.000088,0.010916
2,2017-05-12_cell_2,-0.004487,0.004281,0.000018,-0.017216,0.000013,0.017229
3,2017-05-12_cell_3,-0.007456,0.006007,0.000036,-0.018961,0.000182,0.019143
4,2017-05-12_cell_4,-0.005750,0.004744,0.000023,-0.013958,0.000046,0.014004


### 2.3 Operational Features

초기 100 Cycle의 내부 저항, 온도 및 충전 시간 데이터를 이용하여 배터리의 초기 운용 상태를 나타내는 Feature를 생성한다.

- `mean_IR`: 평균 내부 저항
- `IR_slope`: Cycle에 따른 내부 저항 변화율
- `mean_Tavg`: 평균 온도
- `mean_Tmax`: 평균 최대 온도
- `max_Tmax`: 초기 100 Cycle에서 관측된 최대 온도
- `mean_chargetime`: 평균 충전 시간

In [12]:
def calculate_operational_features(group):

    features = {}

    # -------------------------
    # Internal Resistance
    # -------------------------
    valid_ir = group[group["IR"] > 0]

    if len(valid_ir) >= 2:
        features["mean_IR"] = valid_ir["IR"].mean()

        features["IR_slope"] = np.polyfit(
            valid_ir["cycle"],
            valid_ir["IR"],
            1
        )[0]
    else:
        features["mean_IR"] = np.nan
        features["IR_slope"] = np.nan


    # -------------------------
    # Temperature
    # -------------------------
    valid_tavg = group[group["Tavg"] > 0]["Tavg"]
    valid_tmax = group[group["Tmax"] > 0]["Tmax"]

    features["mean_Tavg"] = (
        valid_tavg.mean()
        if len(valid_tavg) > 0
        else np.nan
    )

    features["mean_Tmax"] = (
        valid_tmax.mean()
        if len(valid_tmax) > 0
        else np.nan
    )

    features["max_Tmax"] = (
        valid_tmax.max()
        if len(valid_tmax) > 0
        else np.nan
    )


    # -------------------------
    # Charging Time
    # -------------------------
    valid_ct = group[group["chargetime"] > 0]["chargetime"]

    features["mean_chargetime"] = (
        valid_ct.mean()
        if len(valid_ct) > 0
        else np.nan
    )

    return pd.Series(features)


operational_features = (
    early_df
    .groupby("cell_id")
    .apply(calculate_operational_features)
    .reset_index()
)

print("Operational Feature shape:", operational_features.shape)

display(operational_features.head())

Operational Feature shape: (141, 7)


,cell_id,mean_IR,IR_slope,mean_Tavg,mean_Tmax,max_Tmax,mean_chargetime
0,2017-05-12_cell_0,0.016643,-6.107912e-07,31.603747,35.376748,35.994705,13.384231
1,2017-05-12_cell_1,0.016922,8.622394e-07,31.330314,34.160655,34.712265,13.374696
2,2017-05-12_cell_10,0.016564,1.634295e-06,33.208151,36.871450,37.614693,11.688040
3,2017-05-12_cell_11,0.016201,-2.694371e-07,32.759323,36.231754,37.128967,11.684114
4,2017-05-12_cell_12,0.016439,8.569436e-07,33.458022,37.448848,38.354816,10.683854


In [13]:
print("결측치 개수:")
display(operational_features.isna().sum())

결측치 개수:


cell_id            0
mean_IR            9
IR_slope           9
mean_Tavg          0
mean_Tmax          0
max_Tmax           0
mean_chargetime    0
dtype: int64

### 2.4 Final Feature Dataset

생성한 QD, ΔQ(V), 운용 조건 Feature를 Cell 단위로 통합하여 모델 학습에 사용할 최종 Feature Dataset을 구축한다.

IR 결측치는 이 단계에서 임의로 대체하지 않고 유지하며, 모델링 단계에서 Train(Batch 1) 데이터만을 기준으로 처리한다.

In [14]:
# Cell 기본 정보 및 Target
cell_info = (
    df[
        [
            "cell_id",
            "batch",
            "cycle_life",
            "charging_policy"
        ]
    ]
    .drop_duplicates(subset="cell_id")
    .reset_index(drop=True)
)

# Feature 통합
feature_df = (
    cell_info
    .merge(
        qd_features,
        on="cell_id",
        how="left"
    )
    .merge(
        delta_qv_features,
        on="cell_id",
        how="left"
    )
    .merge(
        operational_features,
        on="cell_id",
        how="left"
    )
)

print("Final Feature Dataset shape:", feature_df.shape)
print("전체 Cell 수:", feature_df["cell_id"].nunique())

display(feature_df.head())

Final Feature Dataset shape: (141, 19)
전체 Cell 수: 141


,cell_id,batch,cycle_life,charging_policy,mean_QD,std_QD,QD_slope,deltaQ_mean,deltaQ_std,deltaQ_var,deltaQ_min,deltaQ_max,deltaQ_range,mean_IR,IR_slope,mean_Tavg,mean_Tmax,max_Tmax,mean_chargetime
0,2017-05-12_cell_0,2017-05-12,1190.0,3_6C-80PER_3_6C,0.280303,0.017804,-0.000074,-0.002873,0.003109,0.000010,-0.008460,0.000768,0.009228,0.016643,-6.107912e-07,31.603747,35.376748,35.994705,13.384231
1,2017-05-12_cell_1,2017-05-12,1179.0,3_6C-80PER_3_6C,0.280466,0.004920,0.000007,-0.004100,0.003112,0.000010,-0.011004,-0.000088,0.010916,0.016922,8.622394e-07,31.330314,34.160655,34.712265,13.374696
2,2017-05-12_cell_2,2017-05-12,1177.0,3_6C-80PER_3_6C,0.281207,0.004878,0.000032,-0.004487,0.004281,0.000018,-0.017216,0.000013,0.017229,0.016769,1.162834e-06,31.479584,34.544644,35.127342,13.378886
3,2017-05-12_cell_3,2017-05-12,1226.0,4C-80PER_4C,0.286239,0.005598,0.000010,-0.007456,0.006007,0.000036,-0.018961,0.000182,0.019143,0.016311,8.373194e-07,29.942199,30.662263,31.691414,12.053277
4,2017-05-12_cell_4,2017-05-12,1227.0,4C-80PER_4C,0.285034,0.006009,-0.000046,-0.005750,0.004744,0.000023,-0.013958,0.000046,0.014004,0.016669,2.768146e-07,31.448884,34.382516,35.651741,12.053314


## 3. Save Feature Dataset

생성한 Cell 단위 Feature Dataset을 모델링 단계에서 사용할 수 있도록 저장한다.

In [15]:
from pathlib import Path

RESULTS_DIR = Path.cwd().parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)

feature_path = RESULTS_DIR / "feature_dataset.csv"

feature_df.to_csv(feature_path, index=False)

print("저장 완료:", feature_path)
print("Dataset shape:", feature_df.shape)
print("파일 존재:", feature_path.exists())

저장 완료: /Users/seunghyunsuh/Desktop/SKALA 강의자료/12주차(09.28~10.02)/mini project/results/feature_dataset.csv
Dataset shape: (141, 19)
파일 존재: True


### Feature Engineering Summary

초기 100 Cycle 데이터를 기반으로 Cell 단위 Feature Dataset을 구축하였다.

- **Discharge Capacity**: `mean_QD`, `std_QD`, `QD_slope`
- **ΔQ(V)**: `deltaQ_mean`, `deltaQ_std`, `deltaQ_var`, `deltaQ_min`, `deltaQ_max`, `deltaQ_range`
- **Internal Resistance**: `mean_IR`, `IR_slope`
- **Temperature**: `mean_Tavg`, `mean_Tmax`, `max_Tmax`
- **Charging Time**: `mean_chargetime`

`charging_policy`는 Batch 간 프로토콜 표기 방식과 범주가 상이하므로 초기 모델의 입력 Feature에서는 제외하고, 추가 Feature 실험 단계에서 활용 여부를 검토한다.

IR 결측치는 Feature Engineering 단계에서 임의로 대체하지 않으며, 모델링 단계에서 Train(Batch 1) 데이터만을 기준으로 처리한다.

In [16]:
# 임시 확인용 셀

import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocess import load_battery_data

DATA_DIR = "/Users/seunghyunsuh/Downloads/archive"

mats_check = load_battery_data(DATA_DIR)

target_check = []

for filename, mat_data in mats_check.items():

    batch_name = filename.split("_")[0]

    main_key = (
        "batch"
        if "batch" in mat_data
        else [k for k in mat_data.keys() if not k.startswith("__")][0]
    )

    batch_struct = mat_data[main_key]
    cycles = batch_struct["cycles"]
    official_life = batch_struct["cycle_life"]

    for idx, cell in enumerate(cycles):

        len_v = len(cell["V"])
        raw_life = float(np.asarray(official_life[idx]).squeeze())

        target_check.append({
            "cell_id": f"{batch_name}_cell_{idx}",
            "len_V": len_v,
            "official_cycle_life": raw_life,
            "difference": raw_life - len_v
        })

target_check_df = pd.DataFrame(target_check)

print(
    "Target 불일치 Cell 수:",
    (target_check_df["difference"] != 0).sum()
)

display(
    target_check_df[
        target_check_df["cell_id"].isin([
            "2018-02-20_cell_37",
            "2018-02-20_cell_38"
        ])
    ]
)

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (2017-05-12_batchdata_updated_struct_errorcorrect.mat)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (2018-02-20_batchdata_updated_struct_errorcorrect.mat)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)


로딩 중... (2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat)


ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

로딩 중... (2018-04-12_batchdata_updated_struct_errorcorrect.mat)

총 4개 .mat 파일 로딩 완료
Target 불일치 Cell 수: 141


,cell_id,len_V,official_cycle_life,difference
83,2018-02-20_cell_37,101,NaN,NaN
84,2018-02-20_cell_38,101,NaN,NaN


In [17]:
# 원본 cycle_life 구조 확인

for filename, mat_data in mats_check.items():

    main_key = (
        "batch"
        if "batch" in mat_data
        else [k for k in mat_data.keys() if not k.startswith("__")][0]
    )

    cycle_life_raw = mat_data[main_key]["cycle_life"]

    print("\n파일:", filename)
    print("type:", type(cycle_life_raw))
    print("shape:", np.asarray(cycle_life_raw).shape)
    print("앞 10개:", cycle_life_raw[:10])


파일: 2017-05-12_batchdata_updated_struct_errorcorrect.mat
type: <class 'list'>
shape: (46,)
앞 10개: [array(1190.), array(1179.), array(1177.), array(1226.), array(1227.), array(1074.), array(636.), array(870.), array(879.), array(1054.)]

파일: 2018-02-20_batchdata_updated_struct_errorcorrect.mat
type: <class 'list'>
shape: (47,)
앞 10개: [array(477.), array(491.), array(424.), array(499.), array(444.), array(416.), array(393.), array(777.), array(904.), array(791.)]

파일: 2018-04-03_varcharge_batchdata_updated_struct_errorcorrect.mat
type: <class 'list'>
shape: (2,)
앞 10개: [array(nan), array(nan)]

파일: 2018-04-12_batchdata_updated_struct_errorcorrect.mat
type: <class 'list'>
shape: (46,)
앞 10개: [array(1009.), array(1063.), array(1267.), array(1115.), array(1048.), array(828.), array(667.), array(1836.), array(828.), array(1039.)]


In [18]:
batch2_file = "2018-02-20_batchdata_updated_struct_errorcorrect.mat"

batch2_raw = mats_check[batch2_file]["batch"]["cycle_life"]

for idx in [36, 37, 38, 39, 40, 41, 46]:
    print(
        f"cell_{idx}:",
        float(np.asarray(batch2_raw[idx]).squeeze())
    )

cell_36: nan
cell_37: nan
cell_38: nan
cell_39: nan
cell_40: nan
cell_41: 442.0
cell_46: 503.0


In [19]:
# 공식 cycle_life 결측치 확인

missing_targets = []

for filename, mat_data in mats_check.items():

    batch_name = filename.split("_")[0]

    main_key = (
        "batch"
        if "batch" in mat_data
        else [k for k in mat_data.keys() if not k.startswith("__")][0]
    )

    official_life = mat_data[main_key]["cycle_life"]

    for idx, value in enumerate(official_life):

        value = float(np.asarray(value).squeeze())

        if np.isnan(value):
            missing_targets.append({
                "cell_id": f"{batch_name}_cell_{idx}",
                "batch": batch_name
            })

missing_target_df = pd.DataFrame(missing_targets)

print("공식 cycle_life가 NaN인 Cell 수:", len(missing_target_df))
display(missing_target_df)

공식 cycle_life가 NaN인 Cell 수: 12


,cell_id,batch
0,2018-02-20_cell_22,2018-02-20
1,2018-02-20_cell_23,2018-02-20
2,2018-02-20_cell_35,2018-02-20
3,2018-02-20_cell_36,2018-02-20
4,2018-02-20_cell_37,2018-02-20
5,2018-02-20_cell_38,2018-02-20
6,2018-02-20_cell_39,2018-02-20
7,2018-02-20_cell_40,2018-02-20
8,2018-04-03_cell_0,2018-04-03
9,2018-04-03_cell_1,2018-04-03


In [20]:
print("feature_df cycle_life NaN:", feature_df["cycle_life"].isna().sum())

print(
    feature_df.groupby("batch")["cycle_life"]
    .apply(lambda x: x.isna().sum())
)

feature_df cycle_life NaN: 12
batch
2017-05-12    0
2018-02-20    8
2018-04-03    2
2018-04-12    2
Name: cycle_life, dtype: int64
